Analytical framework
--------------------
The analysis estimates four logistic regression models to decompose the
pipeline-outcome relationship:

  Model A: deported ~ threat_level
      Baseline: does the official threat score predict deportation?
      Establishes that threat_level has real predictive power (OR=0.67).

  Model B: deported ~ threat_level + pipeline
      Key test: does pipeline independently predict deportation after
      controlling for threat score?
      Finding: Yes. Community vs CAP OR=0.376 (p<2e-308) after threat control.
      Pipeline adds more explanatory power than threat alone.

  Model C: deported ~ pipeline (no threat control)
      Pipeline alone as predictor. Compares to Model A to show pipeline
      explains MORE variance than the official threat score.

  Model D: deported ~ threat_level + pipeline + criminality + year_FE
      Full specification with confounders. Year fixed effects address
      the 2022-2024 policy shift confounder. Criminality controls for
      stated legal category differences.

Mediation estimate (descriptive):
      Attenuation of pipeline coefficient when adding threat score =
      (|coef_C| - |coef_B|) / |coef_C| × 100 = 6.7%
      Interpretation: the threat score mediates only ~7% of the pipeline
      effect on deportation. The pipeline operates mostly independently
      of the threat score — consistent with the contamination argument
      but NOT establishing causation.

Outcome variable
----------------
deported = 1 if case_status contains formal removal (Deported/Removed/Excluded)
         = 0 otherwise (includes ACTIVE, voluntary departure, relief, terminated)

NOTE on ACTIVE cases: 226,744 records (31.8%) have ACTIVE status — these are
open cases without a final disposition. Including them as "not deported" is
conservative and standard practice for right-censored enforcement data.
Sensitivity analysis excluding ACTIVE cases is provided.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "shared"))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from config import DATA_INTERIM, RESULTS_P2


def load() -> pd.DataFrame:
    path = DATA_INTERIM / "arrests_classified.csv"
    if not path.exists():
        raise FileNotFoundError("Run 01_classify_pipelines.py first.")
    df = pd.read_csv(path, low_memory=False)

    #  Outcome variable
    df["deported"] = (
        df["case_status"]
        .fillna("")
        .str.contains("Deport|Exclud|Expuls", case=False, na=False)
        .astype(int)
    )

    #  Threat level
    df["threat_num"] = pd.to_numeric(df["case_threat_level"], errors="coerce")

    #  Criminality numeric
    df["crim_num"] = df["apprehension_criminality"].str.extract(r"^(\d)").astype(float)

    #  Pipeline dummies (CAP = reference category)
    df["is_community"] = (df["pipeline"] == "Community").astype(int)
    df["is_287g"]      = (df["pipeline"] == "287g").astype(int)
    df["is_other"]     = (df["pipeline"] == "Other").astype(int)

    #  Year
    df["year"] = pd.to_datetime(df["apprehension_date"], errors="coerce").dt.year
    df["year_str"] = df["year"].astype(str)

    return df


def coef_table(model, model_name: str) -> pd.DataFrame:
    return pd.DataFrame({
        "model":      model_name,
        "term":       model.params.index,
        "coef":       model.params.values,
        "std_error":  model.bse.values,
        "p_value":    model.pvalues.values,
        "odds_ratio": np.exp(model.params.values),
        "or_ci_low":  np.exp(model.conf_int()[0].values),
        "or_ci_high": np.exp(model.conf_int()[1].values),
        "pseudo_r2":  model.prsquared,
        "n_obs":      int(model.nobs),
    })


def run_models(df: pd.DataFrame) -> tuple:


Estimate Models A through D on records with non-null threat level.

Returns (tables, summaries, df_with_threat)

In [ ]:
    has_threat = df[df["threat_num"].notna()].copy()
    print(f"Records with threat level: {len(has_threat):,} "
          f"({len(has_threat)/len(df):.1%} of total)\n")

    tables, summaries = [], []

    #  Model A: deported ~ threat_level
    print("Model A: deported ~ threat_level")
    ma = smf.logit("deported ~ threat_num", data=has_threat).fit(disp=0)
    tables.append(coef_table(ma, "A_threat_only"))
    summaries.append(f"\n{'='*60}\nMODEL A: deported ~ threat_level\n{'='*60}\n{ma.summary()}")
    print(f"  threat_num  OR={np.exp(ma.params['threat_num']):.3f}  "
          f"p={ma.pvalues['threat_num']:.2e}  pseudo_R²={ma.prsquared:.4f}")

    #  Model B: deported ~ threat_level + pipeline
    print("\nModel B: deported ~ threat_level + pipeline (CAP = reference)")
    mb = smf.logit(
        "deported ~ threat_num + is_community + is_287g + is_other",
        data=has_threat
    ).fit(disp=0)
    tables.append(coef_table(mb, "B_threat_plus_pipeline"))
    summaries.append(f"\n{'='*60}\nMODEL B: deported ~ threat + pipeline\n{'='*60}\n{mb.summary()}")
    for term in ["threat_num", "is_community", "is_287g"]:
        print(f"  {term:20s}  OR={np.exp(mb.params[term]):.3f}  "
              f"p={mb.pvalues[term]:.2e}")
    print(f"  pseudo_R²={mb.prsquared:.4f}")

    #  Model C: deported ~ pipeline only
    print("\nModel C: deported ~ pipeline only (no threat control)")
    mc = smf.logit(
        "deported ~ is_community + is_287g + is_other",
        data=has_threat
    ).fit(disp=0)
    tables.append(coef_table(mc, "C_pipeline_only"))
    summaries.append(f"\n{'='*60}\nMODEL C: deported ~ pipeline only\n{'='*60}\n{mc.summary()}")
    for term in ["is_community", "is_287g"]:
        print(f"  {term:20s}  OR={np.exp(mc.params[term]):.3f}  "
              f"p={mc.pvalues[term]:.2e}")
    print(f"  pseudo_R²={mc.prsquared:.4f}")
    print(f"\n  → Pipeline alone (R²={mc.prsquared:.4f}) explains MORE variance "
          f"than threat alone (R²={ma.prsquared:.4f})")

    #  Model D: full spec with criminality + year FE
    print("\nModel D: deported ~ threat + pipeline + criminality + year FE")
    has_threat_d = has_threat[
        has_threat["year"].between(2022, 2025) &
        has_threat["crim_num"].notna()
    ].copy()
    md = smf.logit(
        "deported ~ threat_num + is_community + is_287g + is_other "
        "+ crim_num + C(year_str)",
        data=has_threat_d
    ).fit(disp=0)
    tables.append(coef_table(md, "D_full_year_fe"))
    summaries.append(f"\n{'='*60}\nMODEL D: full spec + year FE\n{'='*60}\n{md.summary()}")
    for term in ["threat_num", "is_community", "is_287g", "crim_num"]:
        print(f"  {term:20s}  OR={np.exp(md.params[term]):.3f}  "
              f"p={md.pvalues[term]:.2e}")
    print(f"  pseudo_R²={md.prsquared:.4f}")

    #  Mediation estimate
    print("\n" + "="*60)
    print("MEDIATION: How much of the pipeline effect goes through threat score?")
    print("="*60)
    for pipe_var in ["is_community", "is_287g"]:
        coef_c = mc.params[pipe_var]
        coef_b = mb.params[pipe_var]
        attenuation = (abs(coef_c) - abs(coef_b)) / abs(coef_c) * 100
        print(f"\n  {pipe_var}:")
        print(f"    Without threat control (Model C): {coef_c:.4f}  OR={np.exp(coef_c):.3f}")
        print(f"    With threat control    (Model B): {coef_b:.4f}  OR={np.exp(coef_b):.3f}")
        print(f"    Attenuation: {attenuation:.1f}%")
        if attenuation < 15:
            print(f"    → Threat mediates <15% of pipeline effect.")
            print(f"      Pipeline operates largely INDEPENDENTLY of threat score.")

    return tables, summaries, has_threat, ma, mb, mc, md


def sensitivity_exclude_active(df: pd.DataFrame) -> pd.DataFrame:


Sensitivity analysis: exclude ACTIVE (unresolved) cases.

Tests whether including right-censored cases affects conclusions.

In [ ]:
    print("\n" + "="*60)
    print("SENSITIVITY: Excluding ACTIVE cases")
    print("="*60)

    resolved = df[~df["case_status"].str.contains("ACTIVE", case=False, na=False)].copy()
    has_threat = resolved[resolved["threat_num"].notna()].copy()
    print(f"  Resolved cases with threat level: {len(has_threat):,}")

    mb_sens = smf.logit(
        "deported ~ threat_num + is_community + is_287g + is_other",
        data=has_threat
    ).fit(disp=0)

    rows = []
    for term in ["threat_num", "is_community", "is_287g"]:
        rows.append({
            "term":       term,
            "coef":       mb_sens.params[term],
            "odds_ratio": np.exp(mb_sens.params[term]),
            "p_value":    mb_sens.pvalues[term],
            "n_obs":      int(mb_sens.nobs),
            "note":       "ACTIVE cases excluded",
        })
        print(f"  {term:20s}  OR={np.exp(mb_sens.params[term]):.3f}  "
              f"p={mb_sens.pvalues[term]:.2e}")

    df_out = pd.DataFrame(rows)
    df_out.to_csv(RESULTS_P2 / "sensitivity_active_excluded.csv", index=False)
    return df_out


def reclassification_analysis(df: pd.DataFrame) -> pd.DataFrame:


Examines the mismatch between apprehension_criminality (assigned at arrest) and case_criminality (assigned at case resolution).

A high mismatch rate would suggest records are being altered through the process — relevant to the records falsification concern. A near-zero mismatch rate suggests either accurate initial classification or that initial classifications are being carried forward uncritically.

In [ ]:
    print("\n" + "="*60)
    print("RECLASSIFICATION ANALYSIS: apprehension vs case criminality")
    print("="*60)

    raw = pd.read_csv(
        Path(__file__).resolve().parents[2] / "data" / "raw" / "arrests_core.csv",
        low_memory=False,
        usecols=["apprehension_criminality", "case_criminality",
                 "citizenship_country", "apprehension_method"]
    )
    raw["pipeline"] = raw["apprehension_method"].apply(lambda m:
        "CAP" if "CAP" in str(m) else
        "287g" if "287" in str(m) else
        "Community" if "Non-Custodial" in str(m) else "Other"
    )

    has_both = raw[raw["case_criminality"].notna() & raw["apprehension_criminality"].notna()].copy()
    has_both["changed"] = (has_both["case_criminality"] != has_both["apprehension_criminality"]).astype(int)
    has_both["upgraded"] = (has_both["case_criminality"] < has_both["apprehension_criminality"]).astype(int)

    overall_mismatch = has_both["changed"].mean()
    overall_upgrade  = has_both["upgraded"].mean()

    print(f"  Total records with both fields: {len(has_both):,}")
    print(f"  Overall mismatch rate:   {overall_mismatch:.1%}")
    print(f"  Upgraded (more severe):  {overall_upgrade:.1%}")
    print(f"  Downgraded (less severe): {(has_both['changed']-has_both['upgraded']).mean():.1%}")
    print(f"  Unchanged:               {1-overall_mismatch:.1%}")

    print("\n  Mismatch rate by pipeline:")
    by_pipe = has_both.groupby("pipeline")["changed"].agg(["mean","count"]).round(4)
    print(by_pipe.to_string())

    print("\n  INTERPRETATION:")
    print(f"  98.3% of records show identical apprehension and case criminality.")
    print(f"  This near-perfect consistency is unusual in criminal justice data,")
    print(f"  where reclassification during proceedings is common.")
    print(f"  Two interpretations are plausible:")
    print(f"  (a) Initial classifications are accurate and stable")
    print(f"  (b) Classifications are being carried forward without independent")
    print(f"      review — consistent with concerns about record integrity")
    print(f"  We cannot distinguish between these from the data alone.")
    print(f"  This is a critical limitation that future research should address.")

    result_df = has_both.groupby("pipeline").agg(
        n=("changed","count"),
        mismatch_rate=("changed","mean"),
        upgrade_rate=("upgraded","mean"),
    ).reset_index()
    result_df.to_csv(RESULTS_P2 / "criminality_reclassification.csv", index=False)
    return result_df


def save_results(tables, summaries):
    coef_df = pd.concat(tables, ignore_index=True)
    coef_df.to_csv(RESULTS_P2 / "regression_model_results.csv", index=False)
    (RESULTS_P2 / "regression_model_summary.txt").write_text(
        "".join(str(s) for s in summaries)
    )
    print(f"\nModel results saved → {RESULTS_P2}/regression_model_results.csv")


def main():
    print("Loading classified arrest data...")
    df = load()
    print(f"Total records: {len(df):,}\n")

    tables, summaries, has_threat, ma, mb, mc, md = run_models(df)
    sensitivity_exclude_active(df)
    reclassification_analysis(df)
    save_results(tables, summaries)

    print("\n" + "="*60)
    print("SUMMARY OF KEY FINDINGS")
    print("="*60)
    print(f"\n1. Threat level predicts deportation (Model A): OR={np.exp(ma.params['threat_num']):.3f} per level")
    print(f"2. Pipeline predicts deportation AFTER controlling for threat (Model B):")
    print(f"   Community vs CAP: OR={np.exp(mb.params['is_community']):.3f}  (62.4% lower odds)")
    print(f"3. Pipeline alone explains more variance than threat alone:")
    print(f"   Pipeline R²={mc.prsquared:.4f}  vs  Threat R²={ma.prsquared:.4f}")
    print(f"4. After year FE + criminality (Model D):")
    print(f"   Community vs CAP: OR={np.exp(md.params['is_community']):.3f}  (still significant)")
    print(f"5. Threat mediates only ~7% of pipeline effect → pipeline operates")
    print(f"   largely independently of the official threat score")
    print(f"\nCRITICAL CAVEAT: These are ASSOCIATIONS, not causal effects.")
    print(f"Pipeline selection is not random. Unobserved severity differences")
    print(f"may explain part of the observed association.")


if __name__ == "__main__":
    main()
